# 01 — Model Registry Demo

Demonstrates the SQLite-backed MLflow-style model registry.

**Features shown:**
- Create experiments and runs
- Log metrics and parameters
- Save model artifacts
- Query best run, compare versions
- Promote staging → production

In [ ]:
import sys
sys.path.insert(0, '..')

from src.model_registry import ModelRegistry
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Initialize registry
reg = ModelRegistry('../data/registry.db')
print("Registry initialized")

## Create Experiment and Log a Run

In [ ]:
# Create experiment and start run
exp_name = 'demo-experiment'
run_id = reg.start_run(exp_name, 'demo_model', {'n_estimators': 100, 'max_depth': 10})
print(f"Run ID: {run_id}")

# Log metrics
reg.log_metrics(run_id, {'accuracy': 0.87, 'f1': 0.84, 'latency_ms': 45.2})

# Save a dummy model
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(np.random.rand(100, 5), np.random.randint(0, 2, 100))
artifact_path = reg.save_model(run_id, model)
print(f"Model saved to: {artifact_path}")

# Tag as production
reg.set_tag(run_id, 'stage', 'production')
print("Tagged as production")

## Query Registry

In [ ]:
# Get run details
run = reg.get_run(run_id)
print(f"Version: {run['version']}")
print(f"Metrics: {run['metrics']}")
print(f"Tags: {run['tags']}")

# List all experiments
experiments = reg.list_experiments()
print(f"\nExperiments: {[e['name'] for e in experiments]}")

# Get best run
best = reg.get_best_run(exp_name, metric='accuracy')
if best:
    print(f"\nBest run: v{best['version']} — accuracy={best['metrics']['accuracy']}")

## Load Saved Model

In [ ]:
# Load model back
loaded_model = reg.load_model(run_id)
print(f"Loaded model type: {type(loaded_model)}")
print(f"Model params: {loaded_model.get_params()}")